In [7]:
# ML Experiment Lab
# Reproducible ML experiment and benchmarking system

import os
import time
import json
import random

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("ML Experiment Lab")
print("Environment initialized successfully.")

ML Experiment Lab
Environment initialized successfully.


In [8]:
def validate_dataset(X, y):
    """Validate a dataset before running an ML experiment."""

    errors = []

    # Check for empty dataset
    if len(X) == 0:
        errors.append("Dataset is empty.")

    # Check that features and labels have the same number of rows
    if len(X) != len(y):
        errors.append("Features and labels have different lengths.")

    # Check for missing values
    if X.isnull().any().any():
        errors.append("Dataset contains missing values.")

    # Check for infinite values
    if np.isinf(X.select_dtypes(include=np.number)).any().any():
        errors.append("Dataset contains infinite values.")

    # Check for duplicate rows
    if X.duplicated().any():
        errors.append("Dataset contains duplicate rows.")

    if errors:
        print("❌ Dataset validation failed:")
        for error in errors:
            print(f"  - {error}")
        return False

    print("✅ Dataset validation passed.")
    print(f"   Samples: {len(X)}")
    print(f"   Features: {X.shape[1]}")
    print(f"   Classes: {y.nunique()}")

    return True

In [9]:
# Load example dataset
dataset = load_breast_cancer()

X = pd.DataFrame(
    dataset.data,
    columns=dataset.feature_names
)

y = pd.Series(dataset.target, name="target")

# Validate the dataset
validate_dataset(X, y)

✅ Dataset validation passed.
   Samples: 569
   Features: 30
   Classes: 2


True

In [10]:
def create_preprocessing_pipeline():
    """Create a reusable preprocessing pipeline."""

    preprocessing = Pipeline([
        ("scaler", StandardScaler())
    ])

    return preprocessing


print("✅ Preprocessing pipeline created.")

✅ Preprocessing pipeline created.


In [11]:
# Create preprocessing pipeline
preprocessor = create_preprocessing_pipeline()

print(preprocessor)

Pipeline(steps=[('scaler', StandardScaler())])


In [29]:
def run_experiment(model, model_name, X, y, seed=42):
    """Train and evaluate one ML experiment."""

    logger.info(f"Starting experiment: {model_name} | seed={seed}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    pipeline = Pipeline([
        ("preprocessing", StandardScaler()),
        ("model", model)
    ])

    start_time = time.time()
    pipeline.fit(X_train, y_train)
    training_time = time.time() - start_time

    start_time = time.time()
    predictions = pipeline.predict(X_test)
    inference_time = time.time() - start_time

    results = {
        "model": model_name,
        "seed": seed,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "training_time": training_time,
        "inference_time": inference_time
    }

    log_experiment_result(results)

    logger.info(f"Completed experiment: {model_name} | seed={seed}")

    return results

In [13]:
 = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

print("Available models:")

for model_name in models:
    print(f" - {model_name}")

Available models:
 - Logistic Regression
 - Random Forest


In [14]:
results = []

for model_name, model in models.items():

    result = run_experiment(
        model=model,
        model_name=model_name,
        X=X,
        y=y,
        seed=42
    )

    results.append(result)

results_df = pd.DataFrame(results)

results_df

,model,seed,accuracy,precision,recall,f1,training_time,inference_time
0,Logistic Regression,42,0.982456,0.986111,0.986111,0.986111,0.031585,0.002608
1,Random Forest,42,0.956140,0.958904,0.972222,0.965517,0.329831,0.012451


In [15]:
def run_benchmark(models, X, y, seeds):
    """Run all configured experiments and collect benchmark results."""

    all_results = []

    for model_name, model in models.items():
        for seed in seeds:

            print(f"Running: {model_name} | seed={seed}")

            result = run_experiment(
                model=model,
                model_name=model_name,
                X=X,
                y=y,
                seed=seed
            )

            all_results.append(result)

    return pd.DataFrame(all_results)

In [16]:
seeds = [42, 123, 456]

print(f"Models: {len(models)}")
print(f"Seeds: {len(seeds)}")
print(f"Total experiments: {len(models) * len(seeds)}")

Models: 2
Seeds: 3
Total experiments: 6


In [17]:
benchmark_results = run_benchmark(
    models=models,
    X=X,
    y=y,
    seeds=seeds
)

benchmark_results

Running: Logistic Regression | seed=42
Running: Logistic Regression | seed=123
Running: Logistic Regression | seed=456
Running: Random Forest | seed=42
Running: Random Forest | seed=123
Running: Random Forest | seed=456


,model,seed,accuracy,precision,recall,f1,training_time,inference_time
0,Logistic Regression,42,0.982456,0.986111,0.986111,0.986111,0.024752,0.002852
1,Logistic Regression,123,0.973684,0.972603,0.986111,0.979310,0.019671,0.002601
2,Logistic Regression,456,0.991228,0.986301,1.000000,0.993103,0.015743,0.002985
3,Random Forest,42,0.956140,0.958904,0.972222,0.965517,0.344940,0.009084
4,Random Forest,123,0.973684,0.985915,0.972222,0.979021,0.282212,0.009393
5,Random Forest,456,0.964912,0.959459,0.986111,0.972603,0.278020,0.008357


In [18]:
summary = (
    benchmark_results
    .groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        training_time_mean=("training_time", "mean"),
        inference_time_mean=("inference_time", "mean")
    )
    .reset_index()
)

summary

,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,training_time_mean,inference_time_mean
0,Logistic Regression,0.982456,0.008772,0.981672,0.990741,0.986175,0.020055,0.002813
1,Random Forest,0.964912,0.008772,0.968093,0.976852,0.972380,0.301724,0.008945


In [19]:
experiment_config = {
    "seeds": [42, 123, 456],
    "test_size": 0.2,
    "models": [
        "Logistic Regression",
        "Random Forest"
    ]
}

print("Experiment configuration:")
print(experiment_config)

Experiment configuration:
{'seeds': [42, 123, 456], 'test_size': 0.2, 'models': ['Logistic Regression', 'Random Forest']}


In [20]:
# Save experiment results
benchmark_results.to_csv("benchmark_results.csv", index=False)

# Save benchmark summary
summary.to_csv("benchmark_summary.csv", index=False)

print("✅ Benchmark results saved.")
print("✅ Benchmark summary saved.")

✅ Benchmark results saved.
✅ Benchmark summary saved.


In [30]:
def run_configured_benchmark(config, models, X, y):
    """Run experiments using the provided configuration."""

    selected_models = {
        name: models[name]
        for name in config["models"]
    }

    return run_benchmark(
        models=selected_models,
        X=X,
        y=y,
        seeds=config["seeds"]
    )


print("✅ Configuration-driven benchmark function ready.")

✅ Configuration-driven benchmark function ready.


In [31]:
configured_results = run_configured_benchmark(
    config=experiment_config,
    models=models,
    X=X,
    y=y
)

configured_results

Running: Logistic Regression | seed=42
Running: Logistic Regression | seed=123
Running: Logistic Regression | seed=456
Running: Random Forest | seed=42
Running: Random Forest | seed=123
Running: Random Forest | seed=456


,model,seed,accuracy,precision,recall,f1,training_time,inference_time
0,Logistic Regression,42,0.982456,0.986111,0.986111,0.986111,0.009458,0.002611
1,Logistic Regression,123,0.973684,0.972603,0.986111,0.979310,0.009951,0.002618
2,Logistic Regression,456,0.991228,0.986301,1.000000,0.993103,0.024179,0.003782
3,Random Forest,42,0.956140,0.958904,0.972222,0.965517,0.317976,0.009368
4,Random Forest,123,0.973684,0.985915,0.972222,0.979021,0.305326,0.014676
5,Random Forest,456,0.964912,0.959459,0.986111,0.972603,0.437195,0.013197


In [23]:
configured_results.to_csv("configured_benchmark_results.csv", index=False)

print("✅ Configured benchmark results saved.")

✅ Configured benchmark results saved.


In [24]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("ml_experiment_lab")

logger.info("Experiment logging initialized.")

In [25]:
logger.info("Test log message")
print("Logger test completed.")

Logger test completed.


In [28]:
def log_experiment_result(result):
    """Log the important results from an experiment."""

    logger.info(
        f"Result | model={result['model']} "
        f"| seed={result['seed']} "
        f"| f1={result['f1']:.4f} "
        f"| train_time={result['training_time']:.4f}s "
        f"| inference_time={result['inference_time']:.4f}s"
    )


print("✅ Result logging function ready.")

✅ Result logging function ready.


In [32]:
def validate_dataset_with_errors(X, y):
    """Validate dataset and return detailed validation errors."""

    errors = []

    if len(X) == 0:
        errors.append("Dataset is empty.")

    if len(X) != len(y):
        errors.append("Features and labels have different lengths.")

    if X.isnull().any().any():
        errors.append("Dataset contains missing values.")

    if np.isinf(X.select_dtypes(include=np.number)).any().any():
        errors.append("Dataset contains infinite values.")

    if X.duplicated().any():
        errors.append("Dataset contains duplicate rows.")

    if errors:
        logger.error(
            f"Dataset validation failed with {len(errors)} error(s)."
        )
        return False, errors

    logger.info("Dataset validation passed.")
    return True, []


print("✅ Detailed dataset validation ready.")

✅ Detailed dataset validation ready.


In [33]:
# Create intentionally invalid data
X_bad = X.copy()
X_bad.iloc[0, 0] = np.nan

is_valid, errors = validate_dataset_with_errors(X_bad, y)

print("Valid:", is_valid)
print("Errors:", errors)

ERROR:ml_experiment_lab:Dataset validation failed with 1 error(s).


Valid: False
Errors: ['Dataset contains missing values.']


In [34]:
is_valid, errors = validate_dataset_with_errors(X, y)

print("Valid:", is_valid)
print("Errors:", errors)

Valid: True
Errors: []


In [35]:
from datetime import datetime


def create_experiment_record(result, config):
    """Create a structured record for one experiment."""

    record = {
        "timestamp": datetime.now().isoformat(),
        "model": result["model"],
        "seed": result["seed"],
        "test_size": config["test_size"],
        "accuracy": result["accuracy"],
        "precision": result["precision"],
        "recall": result["recall"],
        "f1": result["f1"],
        "training_time": result["training_time"],
        "inference_time": result["inference_time"]
    }

    return record


print("✅ Experiment record function ready.")

✅ Experiment record function ready.


In [36]:
sample_record = create_experiment_record(
    configured_results.iloc[0].to_dict(),
    experiment_config
)

sample_record

{'timestamp': '2026-09-02T18:53:35.109504',
 'model': 'Logistic Regression',
 'seed': 42,
 'test_size': 0.2,
 'accuracy': 0.9824561403508771,
 'precision': 0.9861111111111112,
 'recall': 0.9861111111111112,
 'f1': 0.9861111111111112,
 'training_time': 0.009457826614379883,
 'inference_time': 0.0026111602783203125}

In [37]:
experiment_records = []

for _, result in configured_results.iterrows():
    record = create_experiment_record(
        result.to_dict(),
        experiment_config
    )
    experiment_records.append(record)

experiment_records_df = pd.DataFrame(experiment_records)

experiment_records_df

,timestamp,model,seed,test_size,accuracy,precision,recall,f1,training_time,inference_time
0,2026-09-02T18:54:39.136300,Logistic Regression,42,0.2,0.982456,0.986111,0.986111,0.986111,0.009458,0.002611
1,2026-09-02T18:54:39.136490,Logistic Regression,123,0.2,0.973684,0.972603,0.986111,0.979310,0.009951,0.002618
2,2026-09-02T18:54:39.136610,Logistic Regression,456,0.2,0.991228,0.986301,1.000000,0.993103,0.024179,0.003782
3,2026-09-02T18:54:39.138836,Random Forest,42,0.2,0.956140,0.958904,0.972222,0.965517,0.317976,0.009368
4,2026-09-02T18:54:39.139093,Random Forest,123,0.2,0.973684,0.985915,0.972222,0.979021,0.305326,0.014676
5,2026-09-02T18:54:39.139221,Random Forest,456,0.2,0.964912,0.959459,0.986111,0.972603,0.437195,0.013197


In [38]:
experiment_records_df.to_csv(
    "experiment_records.csv",
    index=False
)

print("✅ Structured experiment records saved.")

✅ Structured experiment records saved.
